# run_pipeline — Phase 2 driver

Runs Landing → Bronze → Silver only. Gold is Phase 3 work.

In [ ]:
from uuid import uuid4

def ensure_widget(name, default, label):
    try: dbutils.widgets.get(name)
    except Exception: dbutils.widgets.text(name, default, label)

ensure_widget('landing_root','', 'Landing root containing raw/ and dims/')
ensure_widget('catalog_name','main','Unity Catalog name')
ensure_widget('start_date','','Start date YYYY-MM-DD; blank = latest partition')
ensure_widget('end_date','','End date YYYY-MM-DD; blank = start date')
ensure_widget('load_type','incremental','full, incremental, or backfill')
landing_root=dbutils.widgets.get('landing_root').strip()
catalog_name=dbutils.widgets.get('catalog_name').strip()
start_date=dbutils.widgets.get('start_date').strip()
end_date=dbutils.widgets.get('end_date').strip()
load_type=dbutils.widgets.get('load_type').strip()
if not landing_root: raise ValueError('Set landing_root before running the driver.')
batch_id=f'batch_{uuid4().hex}'
arguments={'landing_root':landing_root,'catalog_name':catalog_name,'start_date':start_date,'end_date':end_date,'load_type':load_type,'batch_id':batch_id}

# Setup is idempotent. Each child receives the exact same batch id and parameters.
dbutils.notebook.run('01_setup', 300, arguments)
for notebook, timeout in [('10_bronze_ingest',1800),('20_silver_dims',600),('21_silver_events',1800)]:
    print(f'Running {notebook} for batch {batch_id}')
    dbutils.notebook.run(notebook, timeout, arguments)
print(f'Phase 2 pipeline completed: {batch_id}')
